# Guía desde cero: regresión lineal y métricas de error

Este notebook es una introducción a los conceptos que vamos a usar para resolver los ejercicios de regresión lineal. La idea es entender **qué es cada cosa antes de escribir código que la calcule**.

## Índice

1. Qué es una regresión lineal
2. Qué es un error o residuo
3. MAE — error absoluto medio
4. MSE — error cuadrático medio
5. RMSE — raíz del error cuadrático medio
6. MAPE — error porcentual absoluto medio
7. R² — coeficiente de determinación
8. Correlación de Pearson
9. P-valor
10. Train / test split
11. StandardScaler
12. Multicolinealidad

Ejecuta las celdas una por una en orden. Cada sección tiene primero una explicación con palabras normales y después código que hace visible lo que acabas de leer.

In [ ]:
# Librerías que vamos a usar a lo largo de todo el notebook.
# Las importamos aquí una sola vez y ya quedan disponibles en el resto de celdas.

import numpy as np                   # cálculo numérico: arrays, medias, raíz cuadrada, etc.
import pandas as pd                  # DataFrames (tablas con columnas nombradas)
import matplotlib.pyplot as plt      # gráficos
import seaborn as sns                # gráficos estadísticos más bonitos encima de matplotlib
from scipy import stats              # funciones estadísticas (correlaciones, p-valores)

# Ajuste estético: fondo blanco con rejilla suave para todos los gráficos
sns.set_style('whitegrid')

# Fijamos una semilla aleatoria para que los números aleatorios que generemos
# salgan siempre iguales. Así tus resultados coincidirán con los que explico.
np.random.seed(42)

---

## 1. Qué es una regresión lineal

Imagínate que tienes una nube de puntos en un gráfico. En el eje X pones una variable (por ejemplo, las horas que ha estudiado cada alumno) y en el eje Y otra (la nota que ha sacado). Cada alumno es un punto.

La **regresión lineal** es una técnica que intenta dibujar **la mejor línea recta posible** por esa nube de puntos. ¿Para qué? Para que, cuando aparezca un alumno nuevo y te diga cuántas horas ha estudiado, puedas usar la línea para predecir qué nota sacará.

La ecuación de una recta es:

$$ y = a + b \cdot x $$

- `y` es lo que queremos predecir (la nota).
- `x` es lo que conocemos (las horas de estudio).
- `a` es el **intercepto**: el valor de y cuando x vale 0 (el punto donde la recta cruza el eje vertical).
- `b` es la **pendiente** o **coeficiente**: cuánto sube o baja y por cada unidad que aumenta x.

Vamos a verlo con un ejemplo muy pequeño, inventado, para que se vea bien.

In [ ]:
# Datos de juguete: 5 alumnos, sus horas de estudio y la nota que sacaron.
# Los creamos a mano para tener control total de los números.

horas = np.array([1, 2, 3, 4, 5])         # horas de estudio (variable X)
notas = np.array([2, 4, 5, 4, 6])         # nota obtenida (variable Y)

# Los ponemos en un DataFrame para que se vea bonito
datos = pd.DataFrame({'horas': horas, 'nota': notas})
datos

In [ ]:
# Entrenamos una regresión lineal muy sencilla sobre estos 5 puntos.
# Importamos la clase del modelo.
from sklearn.linear_model import LinearRegression

# X debe ser 2D (matriz: filas = observaciones, columnas = features).
# Por eso hacemos reshape(-1, 1): -1 significa "calcula tú las filas", 1 = una columna.
X = horas.reshape(-1, 1)
y = notas

# Creamos el modelo (todavía vacío, sin entrenar)
modelo = LinearRegression()

# Lo entrenamos: busca la mejor recta que pasa por esos puntos
modelo.fit(X, y)

# Mostramos los valores que ha encontrado
print(f"Intercepto (a): {modelo.intercept_:.4f}")
print(f"Pendiente (b): {modelo.coef_[0]:.4f}")
print(f"Ecuación de la recta: nota = {modelo.intercept_:.2f} + {modelo.coef_[0]:.2f} · horas")

In [ ]:
# Visualizamos los puntos y la recta ajustada

# Primero los puntos
plt.scatter(horas, notas, s=80, color='steelblue', label='Alumnos')

# Ahora la recta: generamos muchos puntos X entre 0 y 6 y calculamos su predicción
x_recta = np.linspace(0, 6, 100).reshape(-1, 1)
y_recta = modelo.predict(x_recta)
plt.plot(x_recta, y_recta, color='red', label='Recta de regresión')

plt.xlabel('Horas de estudio')
plt.ylabel('Nota')
plt.title('Regresión lineal con 5 puntos de juguete')
plt.legend()
plt.show()

**Cómo leer este gráfico:**

- Los 5 puntos azules son los datos reales de los alumnos.
- La línea roja es la "mejor recta" que ha encontrado el modelo para pasar cerca de todos los puntos.
- La recta no pasa por todos los puntos exactamente, y eso está bien. Si pasara por todos sería sospechoso (y con pocos datos, imposible salvo coincidencia).
- Para cualquier valor nuevo de horas (por ejemplo, 3,5 horas), podemos usar la recta para predecir la nota correspondiente.

Cuando hay varias variables X (varias features), la ecuación se extiende:

$$ y = a + b_1 \cdot x_1 + b_2 \cdot x_2 + b_3 \cdot x_3 + \ldots $$

Cada feature tiene su propio coeficiente. Eso se llama **regresión lineal múltiple**. El ejercicio de Diabetes tiene 10 features, así que su modelo tendrá 1 intercepto y 10 coeficientes.

---

## 2. Qué es un error o residuo

Mira el gráfico anterior: la recta no pasa exactamente por los puntos. Hay una distancia vertical entre cada punto real y lo que la recta predice para ese valor de x. Esa distancia es el **error** (también llamado **residuo**) de ese punto.

$$ \text{error}_i = \text{valor real}_i - \text{valor predicho}_i $$

- Si el error es **positivo**, el modelo **subestima** (la realidad es mayor que la predicción).
- Si el error es **negativo**, el modelo **sobreestima** (la realidad es menor que la predicción).
- Si el error es **cero**, el modelo ha acertado exactamente ese punto.

Cada uno de nuestros 5 alumnos tiene su propio error. Vamos a calcularlos.

In [ ]:
# Aplicamos el modelo a las mismas horas de estudio para obtener las predicciones
notas_predichas = modelo.predict(X)

# El error de cada alumno es la diferencia entre real y predicho
errores = notas - notas_predichas

# Montamos una tabla para ver todo junto
tabla = pd.DataFrame({
    'horas': horas,
    'nota_real': notas,
    'nota_predicha': notas_predichas.round(4),
    'error': errores.round(4)
})
tabla

**Observaciones de la tabla:**

- El alumno de 1 hora sacó un 2 real, pero el modelo predice 2,6. Error = −0,6 (el modelo se ha pasado).
- El alumno de 2 horas sacó un 4 real, el modelo predice 3,4. Error = +0,6 (el modelo se ha quedado corto).
- Etc.

Ahora tenemos 5 errores distintos. ¿Cómo resumimos todo eso en un único número para saber "cómo de bueno es el modelo"? Ahí entran las métricas de error que vamos a ver a continuación.

In [ ]:
# Visualizamos los errores en el gráfico: líneas verticales desde cada punto real hasta la recta

plt.scatter(horas, notas, s=80, color='steelblue', label='Nota real')
plt.plot(x_recta, y_recta, color='red', label='Recta de regresión')
plt.scatter(horas, notas_predichas, s=80, color='orange', marker='x', label='Nota predicha')

# Dibujamos una línea vertical de cada punto real a su predicción (el error)
for i in range(len(horas)):
    plt.plot([horas[i], horas[i]], [notas[i], notas_predichas[i]],
             color='gray', linestyle='--')

plt.xlabel('Horas de estudio')
plt.ylabel('Nota')
plt.title('Errores: distancia vertical entre puntos reales y la recta')
plt.legend()
plt.show()

Cada línea gris discontinua es el error de ese alumno. La regresión lineal ha elegido la recta que hace que la suma de todas esas líneas (al cuadrado) sea lo más pequeña posible.

---

## 3. MAE — Mean Absolute Error (Error Absoluto Medio)

**Qué es con palabras:** en promedio, ¿cuánto se equivoca mi modelo en cada predicción, sin importar si se pasa o se queda corto?

**Por qué existe:** si simplemente sumáramos los errores tal cual, los positivos y los negativos se compensarían y la suma daría casi cero. Eso nos engañaría haciéndonos creer que el modelo es perfecto. La solución: convertir todos los errores a positivos con el **valor absoluto** antes de promediar.

**Fórmula:**

$$ MAE = \frac{1}{n} \sum_{i=1}^{n} |error_i| $$

Que leído en cristiano es: suma los valores absolutos de todos los errores y divídelo por el número de errores.

In [ ]:
# Calculamos el MAE paso a paso, sin usar sklearn todavía

# Paso 1: valor absoluto de cada error
errores_abs = np.abs(errores)
print("Errores absolutos:", errores_abs.round(4))

# Paso 2: sumar todos
suma_abs = errores_abs.sum()
print(f"Suma: {suma_abs:.4f}")

# Paso 3: dividir entre el número de errores
n = len(errores_abs)
mae_manual = suma_abs / n
print(f"MAE = {suma_abs:.4f} / {n} = {mae_manual:.4f}")

In [ ]:
# Lo mismo pero usando numpy en una sola línea: media del valor absoluto
mae_numpy = np.mean(np.abs(errores))
print(f"MAE con numpy: {mae_numpy:.4f}")

# Y ahora usando la función de sklearn
from sklearn.metrics import mean_absolute_error
mae_sklearn = mean_absolute_error(notas, notas_predichas)
print(f"MAE con sklearn: {mae_sklearn:.4f}")

# Los tres valores son idénticos. Ahora sabes lo que hace sklearn por dentro.

**Cómo interpretar el MAE:**

Un MAE de 0,4 significa que, en promedio, nuestro modelo se equivoca en 0,4 puntos al predecir la nota de un alumno. Como las notas van de 0 a 10, eso es bastante preciso.

**Ventaja:** es el más fácil de entender. Está en las mismas unidades que lo que predices (si predices metros, el MAE está en metros; si predices euros, en euros).

**Limitación:** no distingue entre equivocarse poco muchas veces o equivocarse mucho pocas veces. Un error grande y un error pequeño pesan proporcionalmente. Esto puede ser un problema si los errores grandes son especialmente graves para tu aplicación.

---

## 4. MSE — Mean Squared Error (Error Cuadrático Medio)

**Qué es con palabras:** en lugar de tomar el valor absoluto del error, lo elevamos al cuadrado antes de promediar. Esto tiene un efecto clave: **castiga mucho más los errores grandes que los pequeños**.

**Por qué existe:** a veces no basta con saber el error medio. Te puede interesar que tu modelo evite a toda costa equivocaciones gordas, aunque eso signifique equivocarse un pelín más en las pequeñas. Elevar al cuadrado hace que un error de 4 pese 16 y un error de 2 pese solo 4. La ratio cambia: aunque el error 4 es solo el doble del error 2 en magnitud, en cuadrado pesa 4 veces más.

**Fórmula:**

$$ MSE = \frac{1}{n} \sum_{i=1}^{n} error_i^2 $$

In [ ]:
# Cálculo paso a paso del MSE

# Paso 1: elevar cada error al cuadrado
errores_cuad = errores ** 2
print("Errores al cuadrado:", errores_cuad.round(4))

# Paso 2: sumar
suma_cuad = errores_cuad.sum()
print(f"Suma: {suma_cuad:.4f}")

# Paso 3: dividir entre n
mse_manual = suma_cuad / n
print(f"MSE = {suma_cuad:.4f} / {n} = {mse_manual:.4f}")

In [ ]:
# Versión numpy en una línea
mse_numpy = np.mean(errores ** 2)
print(f"MSE con numpy: {mse_numpy:.4f}")

# Versión sklearn
from sklearn.metrics import mean_squared_error
mse_sklearn = mean_squared_error(notas, notas_predichas)
print(f"MSE con sklearn: {mse_sklearn:.4f}")

In [ ]:
# Ejemplo visual de por qué el MSE castiga más los errores grandes.
# Comparamos cómo contribuye cada error al MAE y al MSE.

errores_ejemplo = np.array([-0.6, 0.6, 0.8, -0.8, 1.0])

tabla_comp = pd.DataFrame({
    'error': errores_ejemplo,
    'aporte al MAE (|error|)': np.abs(errores_ejemplo),
    'aporte al MSE (error²)': errores_ejemplo ** 2
})
tabla_comp

Fíjate en la última fila: un error de 1,0. Para el MAE aporta 1,0. Para el MSE aporta también 1,0 (porque 1² = 1). Pero si en lugar de 1,0 el error fuera 2,0, al MAE aportaría 2,0 y al MSE aportaría 4,0. El MSE escala el castigo al cuadrado.

**La limitación grande del MSE:** las unidades están al cuadrado. Si predices metros, el MSE está en metros al cuadrado, que no tiene un significado intuitivo. Por eso casi siempre se usa su raíz cuadrada (RMSE), que vemos a continuación.

---

## 5. RMSE — Root Mean Squared Error

**Qué es con palabras:** el MSE, pero con la raíz cuadrada aplicada al final. Eso devuelve el número a las unidades originales.

**Fórmula:**

$$ RMSE = \sqrt{MSE} $$

**Por qué es útil:** combina lo mejor de los dos mundos. Castiga los errores grandes (como el MSE) pero está en unidades interpretables (como el MAE).

**Relación con el MAE:** el RMSE siempre es mayor o igual que el MAE. Si los dos son parecidos, los errores están repartidos de forma uniforme. Si el RMSE es mucho mayor que el MAE, es señal de que hay unos pocos errores muy grandes (outliers) que están tirando del RMSE hacia arriba.

In [ ]:
# Calculamos el RMSE
rmse = np.sqrt(mse_numpy)
print(f"MSE:  {mse_numpy:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"MAE:  {mae_numpy:.4f}")
print()
print(f"Observa: RMSE ({rmse:.4f}) > MAE ({mae_numpy:.4f}), como debe ser.")
print("La diferencia entre RMSE y MAE nos indica si hay errores muy grandes.")

---

## 6. MAPE — Mean Absolute Percentage Error

**Qué es con palabras:** el error medio expresado como **porcentaje del valor real**. Sirve para saber si nos equivocamos mucho o poco en términos relativos.

**Por qué existe:** un error de 10 unidades no significa lo mismo si el valor real es 1000 (te has equivocado un 1%) que si es 50 (te has equivocado un 20%). El MAPE tiene en cuenta la escala.

**Fórmula:**

$$ MAPE = \frac{1}{n} \sum_{i=1}^{n} \left| \frac{error_i}{real_i} \right| \times 100 $$

Leído en cristiano: para cada alumno, divide su error entre su valor real y tómalo en valor absoluto. Haz la media. Multiplica por 100 para que sea porcentaje.

In [ ]:
# Cálculo manual del MAPE

# Paso 1: para cada error, calcular el porcentaje relativo al valor real
porcentajes = np.abs(errores / notas) * 100
print("Porcentaje de error de cada alumno:", porcentajes.round(2))

# Paso 2: media
mape_manual = np.mean(porcentajes)
print(f"MAPE = {mape_manual:.2f} %")

# Versión sklearn (devuelve proporción, no porcentaje: por eso multiplicamos por 100)
from sklearn.metrics import mean_absolute_percentage_error
mape_sklearn = mean_absolute_percentage_error(notas, notas_predichas) * 100
print(f"MAPE con sklearn: {mape_sklearn:.2f} %")

In [ ]:
# Ejemplo pedagógico: dos pacientes con el mismo error absoluto pero distinta escala

ejemplo = pd.DataFrame({
    'paciente': ['A', 'B'],
    'valor_real': [200, 50],
    'valor_predicho': [190, 40],
    'error_absoluto': [10, 10]
})

# El error absoluto es el mismo (10), pero en términos relativos:
ejemplo['error_porcentual'] = (ejemplo['error_absoluto'] / ejemplo['valor_real']) * 100
ejemplo

En el ejemplo anterior los dos errores absolutos son iguales (10), pero el error relativo en el paciente B es mucho mayor (20 %) que en el A (5 %), porque la escala de B es más pequeña. Eso es lo que captura el MAPE.

**Ventaja:** intuitivo y en porcentaje, independiente de la escala.

**Cuidados importantes:**
- Si algún valor real es 0, la división explota (no se puede dividir entre cero).
- Si algún valor real es muy pequeño, el porcentaje se dispara a cifras enormes.
- No es simétrico: subestimar en 10 y sobreestimar en 10 no siempre dan el mismo porcentaje.

---

## 7. R² — Coeficiente de determinación

**Esta es la métrica reveladora.** Las anteriores te dicen cuánto te equivocas en términos absolutos. El R² te dice algo más importante: **si tu modelo sirve para algo o no**.

**La intuición del baseline tonto:**

Imagínate que no tienes ningún modelo. ¿Cuál es la predicción más tonta que podrías hacer? Responder siempre la media. Si la nota media de la clase es 4,2, pues dices 4,2 para todo el mundo, sin mirar las horas de estudio ni nada. Esa es la peor predicción razonable que puedes dar.

El R² compara tu modelo con ese baseline tonto:

- **R² = 1** → modelo perfecto. Acierta todo sin error.
- **R² = 0** → tu modelo predice igual de bien que responder siempre la media. Es decir, no aporta nada.
- **R² entre 0 y 1** → tu modelo es mejor que la media. R² = 0,7 significa que explicas el 70 % de la variabilidad.
- **R² negativo** → tu modelo es peor que responder siempre la media. Has hecho un modelo que perjudica.

**Fórmula** (no hace falta que te la sepas, pero para entender):

$$ R^2 = 1 - \frac{\text{suma de errores}^2 \text{ del modelo}}{\text{suma de errores}^2 \text{ usando la media como predicción}} $$

In [ ]:
# Calculamos el R² paso a paso para ver qué hay dentro

# Paso 1: errores cuadrados de nuestro modelo
ss_modelo = np.sum((notas - notas_predichas) ** 2)
print(f"Suma de errores² del modelo: {ss_modelo:.4f}")

# Paso 2: errores cuadrados del baseline tonto (predecir siempre la media)
media_notas = notas.mean()
ss_baseline = np.sum((notas - media_notas) ** 2)
print(f"Nota media: {media_notas:.2f}")
print(f"Suma de errores² del baseline (predecir la media): {ss_baseline:.4f}")

# Paso 3: aplicar la fórmula del R²
r2_manual = 1 - (ss_modelo / ss_baseline)
print(f"R² = 1 - ({ss_modelo:.4f} / {ss_baseline:.4f}) = {r2_manual:.4f}")

# Versión sklearn
from sklearn.metrics import r2_score
r2_sklearn = r2_score(notas, notas_predichas)
print(f"R² con sklearn: {r2_sklearn:.4f}")

R² ≈ 0,63 significa que nuestro modelo explica aproximadamente el 63 % de la variabilidad de las notas a partir de las horas de estudio. El 37 % restante lo explican otros factores que no estamos midiendo (nivel previo, motivación, descanso, etc.).

Ahora veamos un caso donde el R² sale negativo: un modelo que predice peor que la media.

In [ ]:
# Ejemplo de modelo malísimo: predecimos a propósito lo contrario
# Si las notas son [2, 4, 5, 4, 6] y su media es 4.2,
# un modelo que predice al revés va a ser peor que predecir siempre 4.2

predicciones_malisimas = np.array([7, 5, 4, 5, 3])  # predicciones inventadas, invertidas

r2_malisimo = r2_score(notas, predicciones_malisimas)
print(f"Notas reales:      {notas}")
print(f"Predicciones:      {predicciones_malisimas}")
print(f"R² del modelo malo: {r2_malisimo:.4f}")
print()
print("R² negativo significa: mejor habríamos hecho prediciendo la media para todos.")

**Esto es exactamente lo que pasa en el ejercicio de Height con edad vs altura:** el R² sale negativo. La razón es que la edad no tiene una relación real con la altura en adultos, así que nuestra recta introduce ruido en las predicciones. Más nos valdría responder "1,73 m" (la media) para todo el mundo.

**Conclusión práctica:** siempre que entrenes un modelo, mira el R² en test. Si sale cerca de 0 o negativo, tu modelo no sirve.

---

## 8. Correlación de Pearson

**Qué es con palabras:** un número entre -1 y +1 que te dice si dos variables tienden a moverse juntas de forma lineal.

- **r = +1**: correlación positiva perfecta. Cuando X sube, Y sube exactamente en línea recta.
- **r = 0**: no hay relación lineal.
- **r = −1**: correlación negativa perfecta. Cuando X sube, Y baja en línea recta.
- **r = 0,8**: correlación positiva fuerte.
- **r = 0,3**: correlación positiva débil.

**Ejemplos intuitivos:**
- Altura y peso en adultos: r ≈ 0,7 (los más altos tienden a pesar más, aunque con variabilidad).
- Horas de sol y consumo de helados: r positiva alta.
- Precio de un producto y demanda: r negativa (a más precio, menos demanda).
- Número de cigüeñas y número de bebés nacidos: r positiva alta pero **no hay relación causal**, ambas cosas correlacionan con algo común (zonas rurales vs urbanas). La correlación no implica causalidad.

**r al cuadrado (r²):** si elevas la correlación al cuadrado, obtienes el porcentaje de variabilidad explicada. Si r = 0,7, entonces r² = 0,49 → X explica el 49 % de la variabilidad de Y.

Vamos a generar tres nubes de puntos con correlaciones distintas para que lo veas.

In [ ]:
# Generamos tres nubes de puntos con distintas correlaciones

n_puntos = 100
x_datos = np.random.normal(0, 1, n_puntos)  # 100 puntos aleatorios con distribución normal

# Caso 1: correlación positiva fuerte (y es casi x + un poco de ruido)
y_positiva = x_datos + np.random.normal(0, 0.3, n_puntos)

# Caso 2: sin correlación (y es ruido puro, no depende de x)
y_nula = np.random.normal(0, 1, n_puntos)

# Caso 3: correlación negativa fuerte
y_negativa = -x_datos + np.random.normal(0, 0.3, n_puntos)

# Calculamos las correlaciones
r_pos, _ = stats.pearsonr(x_datos, y_positiva)
r_nul, _ = stats.pearsonr(x_datos, y_nula)
r_neg, _ = stats.pearsonr(x_datos, y_negativa)

# Los dibujamos uno al lado del otro
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].scatter(x_datos, y_positiva, alpha=0.6)
axes[0].set_title(f'Correlación positiva fuerte\nr = {r_pos:.3f}')
axes[0].set_xlabel('x'); axes[0].set_ylabel('y')

axes[1].scatter(x_datos, y_nula, alpha=0.6, color='gray')
axes[1].set_title(f'Sin correlación\nr = {r_nul:.3f}')
axes[1].set_xlabel('x'); axes[1].set_ylabel('y')

axes[2].scatter(x_datos, y_negativa, alpha=0.6, color='darkred')
axes[2].set_title(f'Correlación negativa fuerte\nr = {r_neg:.3f}')
axes[2].set_xlabel('x'); axes[2].set_ylabel('y')

plt.tight_layout()
plt.show()

- En la nube de la izquierda se ve cómo los puntos forman una línea ascendente: correlación positiva.
- En la del medio los puntos están esparcidos sin patrón: no hay correlación lineal.
- En la de la derecha los puntos forman una línea descendente: correlación negativa.

**Limitación importante de Pearson:** solo detecta relaciones **lineales**. Si dos variables tienen una relación no lineal (por ejemplo, una curva en forma de U), Pearson puede dar cero aunque la relación exista.

---

## 9. P-valor: ¿es fiable la correlación que veo?

**El problema:** si tiras un dado pocas veces, puede salirte un patrón extraño por pura casualidad. Con pocos datos, cualquier correlación que calcules podría ser puro azar.

**Qué es el p-valor con palabras:** la probabilidad de observar los datos que tienes (o unos aún más extremos) si en realidad **no hubiera ninguna relación** entre X e Y.

- **p-valor pequeño (< 0,05)**: es poco probable que lo que ves sea casualidad. La correlación es **estadísticamente significativa**. Te la puedes creer.
- **p-valor grande (> 0,05)**: podría ser casualidad perfectamente. No puedes afirmar que exista relación.

El umbral de 0,05 es una **convención**, no una ley física. Algunos estudios usan 0,01 para ser más estrictos.

Vamos a ver con un ejemplo cómo el p-valor depende del número de observaciones.

In [ ]:
# Mismo fenómeno (correlación real de aproximadamente 0.2) con distintos tamaños de muestra
# Vemos que el p-valor cambia mucho según cuántos datos tengamos

np.random.seed(0)

for n_muestra in [20, 100, 1000]:
    x_m = np.random.normal(0, 1, n_muestra)
    # y tiene una correlación real con x pero débil (mucho ruido)
    y_m = 0.2 * x_m + np.random.normal(0, 1, n_muestra)
    r, p = stats.pearsonr(x_m, y_m)
    significativo = "SÍ" if p < 0.05 else "NO"
    print(f"n = {n_muestra:4d}   r = {r:+.3f}   p-valor = {p:.4f}   ¿significativo? {significativo}")

**Fíjate en lo que ocurre:** la correlación real entre x e y es la misma en los tres casos (tiene un componente de 0,2 x + ruido). Pero:

- Con **20 datos**, el p-valor suele ser alto → no puedes demostrar que hay relación, aunque la haya.
- Con **100 datos**, el p-valor baja.
- Con **1000 datos**, el p-valor es claramente < 0,05 → puedes afirmar con confianza que hay relación.

**Conclusión:** con pocos datos, aunque veas correlación, no puedes fiarte de ella. Esto es exactamente lo que pasa en el ejercicio de Height: 21 alumnos son muy pocos, y el p-valor de la correlación edad-altura sale 0,14 (por encima de 0,05). No podemos afirmar que haya relación.

---

## 10. Train / Test Split

**El problema que resuelve:** si entreno un modelo con todos mis datos y luego mido el error con esos mismos datos, estoy haciendo trampa. El modelo puede **memorizar** los datos en lugar de aprender patrones generales. Me diría que acierta mucho, pero con datos nuevos fallaría.

Este fenómeno se llama **overfitting** (sobreajuste): el modelo se aprende los datos de memoria, incluido su ruido, y no generaliza.

**La solución:** partir los datos en dos grupos:
- **Train (entrenamiento):** el modelo los usa para aprender. Suele ser el 70-80 % de los datos.
- **Test (prueba):** el modelo no los ve durante el entrenamiento. Después lo evaluamos con ellos.

Si el modelo funciona bien en test (no solo en train), sabemos que ha aprendido patrones reales, no memorizado.

Vamos a ver un ejemplo.

In [ ]:
# Creamos un dataset más grande para el ejemplo
np.random.seed(42)
n_total = 50
x_todo = np.linspace(0, 10, n_total)
# La relación real es y = 2x + 1 + ruido
y_todo = 2 * x_todo + 1 + np.random.normal(0, 2, n_total)

# Usamos train_test_split de sklearn
from sklearn.model_selection import train_test_split

# Partimos: 80 % train, 20 % test. random_state=42 fija la aleatoriedad
X_train, X_test, y_train, y_test = train_test_split(
    x_todo.reshape(-1, 1), y_todo,
    test_size=0.2,
    random_state=42
)

print(f"Datos totales:  {n_total}")
print(f"Datos de train: {len(X_train)}")
print(f"Datos de test:  {len(X_test)}")

In [ ]:
# Entrenamos el modelo SOLO con los datos de train
mod = LinearRegression()
mod.fit(X_train, y_train)

# Evaluamos en ambos conjuntos
pred_train = mod.predict(X_train)
pred_test = mod.predict(X_test)

print("Rendimiento en TRAIN (los datos que vio al aprender):")
print(f"  R² = {r2_score(y_train, pred_train):.4f}")
print(f"  MAE = {mean_absolute_error(y_train, pred_train):.4f}")
print()
print("Rendimiento en TEST (datos que NO vio al aprender):")
print(f"  R² = {r2_score(y_test, pred_test):.4f}")
print(f"  MAE = {mean_absolute_error(y_test, pred_test):.4f}")
print()
print("Si train y test dan resultados parecidos, el modelo generaliza bien.")
print("Si train es mucho mejor que test, hay overfitting.")

**Regla práctica:** siempre evalúa tu modelo en test, nunca en train. Los números de train son demasiado optimistas porque el modelo ya ha visto esos datos.

---

## 11. StandardScaler (escalado estándar)

**El problema que resuelve:** imagínate que tienes dos features de escalas muy distintas. Edad (entre 20 y 50) e ingresos anuales (entre 20.000 y 100.000). Un coeficiente de la regresión para edad y otro para ingresos no se pueden comparar directamente: están en unidades distintas y con rangos muy distintos.

**Qué hace StandardScaler:** transforma cada feature para que tenga:
- Media = 0
- Desviación típica = 1

**Fórmula:**

$$ x_{\text{escalada}} = \frac{x_{\text{original}} - \text{media}}{\text{desviación típica}} $$

**Ejemplo intuitivo:** si la edad media en tu grupo es 32 con desviación típica 6:
- Alguien de 38 años → (38 − 32) / 6 = +1 (una desviación típica por encima de la media)
- Alguien de 26 años → (26 − 32) / 6 = −1 (una desviación típica por debajo)
- Alguien de 32 años → 0 (justo en la media)

Ahora la edad está en "desviaciones típicas" en lugar de "años". Y si haces lo mismo con los ingresos, ambas features están en la misma escala y se pueden comparar.

In [ ]:
# Creamos dos variables con escalas muy distintas
edades = np.array([25, 30, 35, 40, 45, 50, 28, 33, 38, 42])
ingresos = np.array([20000, 30000, 45000, 60000, 80000, 95000, 25000, 38000, 55000, 70000])

datos_escalar = pd.DataFrame({'edad': edades, 'ingresos': ingresos})
print("Datos originales:")
print(datos_escalar)
print()
print("Estadísticas antes del escalado:")
print(datos_escalar.describe().loc[['mean', 'std']].round(2))

In [ ]:
# Aplicamos StandardScaler
from sklearn.preprocessing import StandardScaler

# Creamos el escalador
scaler = StandardScaler()

# Lo ajustamos a los datos (aprende la media y la desviación típica)
# y al mismo tiempo los transforma
datos_escalados = scaler.fit_transform(datos_escalar)

# Lo convertimos en DataFrame para verlo bonito
df_escalado = pd.DataFrame(datos_escalados, columns=['edad', 'ingresos'])
print("Datos después del escalado:")
print(df_escalado.round(3))
print()
print("Estadísticas después del escalado:")
print(df_escalado.describe().loc[['mean', 'std']].round(3))
print()
print("Ahora ambas columnas tienen media ≈ 0 y desviación típica ≈ 1.")

**Punto importante para la regresión lineal simple:** escalar las variables **no cambia las predicciones ni las métricas de error**. Solo cambia la escala de los coeficientes del modelo. ¿Entonces por qué se hace?

1. Para **comparar la importancia de las features**. Con los datos sin escalar, un coeficiente grande puede serlo simplemente porque la variable está en una escala grande. Con los datos escalados, cada coeficiente representa el efecto de una desviación típica, así que son directamente comparables.
2. Para otros modelos (Ridge, Lasso, redes neuronales, k-NN) el escalado **sí afecta al rendimiento**, y mucho.

---

## 12. Multicolinealidad

**Qué es con palabras:** cuando dos o más features dicen prácticamente lo mismo (correlacionan mucho entre sí).

**Ejemplo extremo para entenderlo:** imagina que tienes una base de datos con altura en metros y altura en centímetros. Son exactamente la misma información dos veces. Correlación perfecta.

**Por qué es un problema:** la regresión lineal asigna un coeficiente a cada feature. Si dos features son casi iguales, el modelo no sabe si "el efecto" viene de una o de la otra, así que reparte los coeficientes de forma errática. Puedes acabar con coeficientes enormes y opuestos que se cancelan entre sí. El modelo se vuelve:
- Inestable (pequeños cambios en los datos cambian mucho los coeficientes).
- Difícil de interpretar (no puedes decir "esta feature aporta X" con seguridad).

**En el ejercicio de Diabetes:** las variables s1 (colesterol total) y s2 (colesterol LDL) correlacionan 0,90. Tiene sentido biológico: el LDL es una parte del colesterol total, así que cuando sube uno sube el otro. Son casi redundantes. Una buena práctica es quitar una de ellas.

Vamos a verlo.

In [ ]:
# Simulamos un caso extremo: dos features prácticamente idénticas
np.random.seed(0)

n = 200
feature_A = np.random.normal(50, 10, n)
feature_B = feature_A + np.random.normal(0, 0.5, n)  # casi igual a A, con un poquito de ruido
feature_C = np.random.normal(30, 5, n)                # esta sí es independiente

# El target depende solo de A y C (B no aporta nada nuevo)
target = 2 * feature_A + 3 * feature_C + np.random.normal(0, 2, n)

df_multi = pd.DataFrame({'A': feature_A, 'B': feature_B, 'C': feature_C, 'target': target})
print("Matriz de correlación:")
print(df_multi.corr().round(3))

In [ ]:
# Entrenamos dos modelos:
# Modelo 1: con A, B y C (con multicolinealidad)
# Modelo 2: con A y C (sin multicolinealidad)

X1 = df_multi[['A', 'B', 'C']]
X2 = df_multi[['A', 'C']]
y_multi = df_multi['target']

mod1 = LinearRegression().fit(X1, y_multi)
mod2 = LinearRegression().fit(X2, y_multi)

print("Modelo 1 (con A, B, C — hay multicolinealidad entre A y B):")
for name, c in zip(X1.columns, mod1.coef_):
    print(f"  Coef de {name}: {c:+.4f}")
print(f"  R² = {mod1.score(X1, y_multi):.4f}")
print()
print("Modelo 2 (solo A y C — sin multicolinealidad):")
for name, c in zip(X2.columns, mod2.coef_):
    print(f"  Coef de {name}: {c:+.4f}")
print(f"  R² = {mod2.score(X2, y_multi):.4f}")

**Qué observar:**

- En el modelo 2, el coeficiente de A sale cerca de 2 y el de C cerca de 3, que es lo que generamos en los datos. Todo limpio.
- En el modelo 1 (con B añadida), los coeficientes de A y B se vuelven locos porque el modelo no sabe cómo repartir el efecto entre dos variables casi idénticas.
- El R² es prácticamente igual en los dos modelos. Añadir B no ha aportado nada.

**Conclusión práctica:** antes de entrenar, mira la matriz de correlaciones. Si ves parejas de features con correlación > 0,8 o 0,9, considera quitar una.

---

## Resumen final

Hasta aquí los conceptos básicos. Te hago un mapa mental rápido de cómo se conectan:

**Antes de modelar:**
1. Mira los datos con `describe()` y gráficos.
2. Calcula correlaciones de Pearson entre features y target → pistas de qué va a funcionar.
3. Mira el p-valor → ¿tengo datos suficientes para fiarme?
4. Detecta multicolinealidad entre features → plantea quitar alguna.

**Al modelar:**
5. Parte los datos en train/test para evitar overfitting.
6. Entrena `LinearRegression` sobre train.
7. Opcional: `StandardScaler` para poder comparar coeficientes.

**Al evaluar:**
8. Predice sobre test.
9. Calcula MAE, MSE, RMSE, MAPE para cuantificar el error.
10. Calcula **R²**: la métrica clave que te dice si el modelo sirve o no.

**Cheat sheet de métricas:**

| Métrica | Unidades | Qué mide | Cuándo usarla |
|---------|----------|----------|---------------|
| MAE  | Las del target | Error medio en valor absoluto | Comunicación intuitiva |
| MSE  | Unidades² | Error medio al cuadrado (castiga lo grande) | Comparar modelos |
| RMSE | Las del target | Raíz del MSE (castiga lo grande pero interpretable) | Reporte final |
| MAPE | % | Error medio relativo | Comparar problemas con distintas escalas |
| R²   | Adimensional | ¿Mejor que responder la media? | Validar que el modelo sirve |

Con esto ya puedes entender línea por línea los dos ejercicios (Height y Diabetes) y defenderlos como tuyos.